# Convert HOT's Freetown trees to shards

Run on a CPU runtime. Set `REPO_SHA` in the next cell, then run the cells in order.

The labels, tile table and image parquets are downloaded from Hugging Face (pinned at `REVISION`) to `/content`; the images are streamed one record batch at a time, so they are never all in memory. Conversion never reads through the Drive mount: only the finished shards are written back to Drive.

Only HOT `train` and `validation` are converted (as `train` and `val`); HOT `test` is read straight from Hugging Face by the label study.

The repo is private: add a read-only GitHub token (Contents: read) under Colab Secrets (the key icon) as `GITHUB_TOKEN`, with notebook access on.

In [ ]:
SOURCE = "hot"
REPO_SHA = ""
RESOLVE_ZIP = "raw/resolve/Ecoregions2017.zip"  # relative to the Drive root
BLOCK_TILES = 16  # tiles per block side: one region per connected run of tiles in a block

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# the repo is private: a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install /content/fair_trees -r /content/fair_trees/experiments/requirements.txt
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import shutil
import zipfile

from huggingface_hub import hf_hub_download

from experiments.label_study.hot import REPO_ID, REVISION, read_hot

FILES = {
    "labels": "stac/project-50661/labels.parquet",
    "tiles": "stac/project-50661/tiles.parquet",
}
IMAGES = {
    "train": ["data/train-00000-of-00002.parquet", "data/train-00001-of-00002.parquet"],
    "validation": ["data/validation-00000-of-00001.parquet"],
}


def fetch(path):
    return Path(
        hf_hub_download(REPO_ID, path, repo_type="dataset", revision=REVISION, local_dir="/content/hot_raw")
    )


labels_path, tiles_path = fetch(FILES["labels"]), fetch(FILES["tiles"])
image_files = {split: [fetch(p) for p in paths] for split, paths in IMAGES.items()}
data = read_hot(labels_path, tiles_path)
for split, paths in image_files.items():
    print(split, [f"{p.name}: {p.stat().st_size / 1e6:.0f} MB" for p in paths])

resolve_dir = Path("/content/resolve")
resolve_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE / RESOLVE_ZIP) as z:
    z.extractall(resolve_dir)
print(sorted(p.name for p in resolve_dir.rglob("*.shp")))

In [ ]:
import pyarrow.parquet as pq

from experiments.biome import resolve_biome_fn
from experiments.converters.hot import convert


def images(split):
    """Yield (tile_id, image bytes) for every tile of a HOT split, one record batch at a time."""
    for path in image_files[split]:
        for batch in pq.ParquetFile(path).iter_batches(columns=["tile_id", "image"], batch_size=64):
            for row in batch.to_pylist():
                yield row["tile_id"], row["image"]["bytes"]


shapefile = next(resolve_dir.rglob("*.shp"))
converted = Path("/content/converted") / SOURCE
# scratch output on the Colab VM only; convert refuses a non-empty output directory
shutil.rmtree(converted, ignore_errors=True)
stats = convert(data, images, converted, biome_fn=resolve_biome_fn(shapefile), block_tiles=BLOCK_TILES)
print(stats)

In [ ]:
from experiments.shards import manifest_hash, write_shards

shard_dir = DRIVE / "shards" / SOURCE
if (shard_dir / "manifest.json").exists():
    # keep the previous conversion (the runs trained on it record its manifest) instead of writing over it
    old = shard_dir.with_name(f"{SOURCE}-{manifest_hash(shard_dir / 'manifest.json')[:8]}")
    shard_dir.rename(old)
    print("previous shards moved to", old)
write_shards(converted, shard_dir, source=SOURCE, converter_sha=REPO_SHA)
print("manifest sha256:", manifest_hash(shard_dir / "manifest.json"))
shutil.copyfile(converted / "stats.json", shard_dir / "stats.json")

In [ ]:
import random

from affine import Affine
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageDraw

from fair_trees.dataset import labels_to_boxes, load_geojson
from fair_trees.georef import tile_bounds_3857
from fair_trees.preprocess import CHIP_SUFFIXES, read_chip

THUMB = 256
chip_paths = sorted(p for p in converted.glob("*/z*/chips/*") if p.suffix.lower() in CHIP_SUFFIXES)
picked = random.Random(0).sample(chip_paths, min(16, len(chip_paths)))
contact = PILImage.new("RGB", (4 * THUMB, 4 * THUMB))
for i, path in enumerate(picked):
    rgb, _, (x, y, z) = read_chip(path, use_sidecar=False)
    minx, miny, maxx, maxy = tile_bounds_3857(x, y, z)
    res = (maxx - minx) / rgb.shape[1]
    transform = Affine(res, 0, minx, 0, -res, maxy)
    region = path.parents[2]
    tile = PILImage.fromarray(rgb[..., :3])
    draw = ImageDraw.Draw(tile)
    # ignore features (unknown-diameter crowns, other-split crowns and groups) in grey,
    # tree labels in red, tree_group labels in green
    layers = (
        ("ignore", 1, (128, 128, 128)),
        ("labels", 1, (255, 0, 0)),
        ("labels", 2, (0, 255, 0)),
    )
    for name, label, colour in layers:
        geojson = load_geojson(region / f"{name}.geojson")
        if name == "ignore":  # ignore features carry no label: read them as trees
            geojson = {"features": [{**f, "properties": {}} for f in geojson["features"]]}
        for x0, y0, x1, y1 in labels_to_boxes(geojson, transform, label):
            draw.rectangle((x0, y0, x1, y1), outline=colour, width=2)
    contact.paste(tile.resize((THUMB, THUMB)), ((i % 4) * THUMB, (i // 4) * THUMB))
sheet = shard_dir / "qa-contact-sheet.png"
contact.save(sheet)
display(Image(filename=str(sheet)))

In [ ]:
import time

largest = max((p for p in shard_dir.iterdir() if p.suffix == ".tar"), key=lambda p: p.stat().st_size)
start = time.monotonic()
shutil.copyfile(largest, Path("/content") / largest.name)
elapsed = time.monotonic() - start
print(f"{largest.name}: {largest.stat().st_size / 1e6 / elapsed:.1f} MB/s")
# Record this in experiments/DATA_SOURCES.md; restaging must take under 10 minutes.

In [ ]:
drive.flush_and_unmount()